## <b>Lecture09 프롬프트 </b>

### 목표
 - In context learning, COT(Chain of Thought), Few-shot COT을 실습하고 이해해 본다.

## Groq API

Groq API를 활용하여 실습을 진행할 예정입니다.

아래의 링크를 따라 들어가서 직접 API key를 발급 받으시면 됩니다.

1. 웹사이트에 방문해서 회원가입을 진행해 주세요! (https://console.groq.com/home)

2. API key를 발급받은 후 아래의 코드에 key를 복사해서 넣어주세요! (https://console.groq.com/keys)

In [ ]:
!pip install datasets
!pip install gym
!pip install requests
!pip install bs4
!pip install -U datasets huggingface_hub fsspec
!pip install -U langchain langchain-core langchain-community langchain-groq groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 774.9/774.9 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.9/203.9 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 20.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.23.0
    Uninstalling huggingface_hub-1.23.0:
      Successfully uninstalled huggingface_hub-1.23.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: 

In [ ]:
GROQ_API_KEY=""

In [ ]:
from groq import Groq

client = Groq(
    api_key = GROQ_API_KEY
)

# Chat Completion & LangChain

*Chat Completion?
- LLM을 대화 형식으로 호출하기 위한 interface
- 누가 어떤 말을 했는 지를 모델에게 잘 전달하기 위해서 사용

*구성요소?
- System Message: 모델에게 역할, 말투, 지침, 행동 제약 등을 설정하기 위한 메시지  
ex. "You are a helpful and concise assistant."

- User Message: 실제 사용자의 질문, 명령, 입력을 담는 메시지  
ex. "파이썬으로 정렬하는 법 알려줘"

- Assistant Message: 모델의 이전 응답을 나타내는 메시지 (chat history 관리 등을 위해 쓰임)  
ex. 모델의 이전 응답을 나타내는 메시지"

In [ ]:
# *예시
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "수학 문제를 도와줘"},
    {"role": "assistant", "content": "물론이죠! 어떤 문제인가요?"}
]

In [ ]:
system_message = "You are senior math solve." # 어시스턴트의 행동을 설정하고, 대화 동안 어떻게 행동할 지에 대한 특정 명령어를 제공
human_message = "Could suggest one sentence that could help student who solve problem. Too complex instruction make the students confused. so offer simple sentence." # 어시스턴트가 응답해야 하는 사용자의 메세지 설정 (보통 저희가 ChatGPT에 작성하는 프롬프트라고 생각하시면 간단합니다)

chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": system_message
        },
        {
            "role": "user",
            "content": human_message,
        }
    ],

    # 응답을 생성할 모델을 지정하는 과정
    model="llama-3.1-8b-instant", # 오른쪽에 있는 사이트에 나와 있는 모델들로 변경해 가면서 사용할 수 있습니다. https://console.groq.com/docs/rate-limits
    temperature=0 # 답변의 랜덤성을 통제 (0에 가까울수록 실행마다 비슷한 답변을 내놓습니다. 최대 2까지 설정 가능합니다.)
)

print(chat_completion)

ChatCompletion(id='chatcmpl-bd38c646-593b-43c2-a2a6-bc383b1e5a13', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='"To solve a problem, break it down into smaller, manageable steps, and focus on one step at a time."', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=None))], created=1785392637, model='llama-3.1-8b-instant', object='chat.completion', mcp_list_tools=None, service_tier='on_demand', system_fingerprint='fp_020e283281', usage=CompletionUsage(completion_tokens=24, prompt_tokens=66, total_tokens=90, completion_time=0.031881906, completion_tokens_details=None, prompt_time=0.003638858, prompt_tokens_details=None, queue_time=0.060125716, total_time=0.035520764), usage_breakdown=None, x_groq=XGroq(id='req_01kyrv3969ewmtdct7ayebqfht', debug=None, seed=1440897816, usage=None))


In [ ]:
### LLM의 응답만 처리해서 출력해 보기
### chat_completion의 dictionary 형태를 자세히 보면 알 수 있습니다!
print(chat_completion.choices[0].message.content)

"To solve a problem, break it down into smaller, manageable steps, and focus on one step at a time."


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

chat = ChatGroq(temperature=0, model_name="llama-3.1-8b-instant", api_key=GROQ_API_KEY)

### TODO ###
system = "You are major in math"
human = "{text}"
prompt_template = ChatPromptTemplate.from_messages([("system", system), ("human", human)])

chain = prompt_template | chat
response = chain.invoke({"text": "Explain the history of Jinhae."})

In [ ]:
print(response.content)

Jinhae is a city located in the South Gyeongsang Province of South Korea. It has a rich history dating back to the Silla Dynasty (57 BC - 935 AD). 

Historically, Jinhae was known as Jinhae-gu, which translates to 'the place where the sea and the land meet'. It was a strategic location for trade and commerce due to its proximity to the sea and its natural harbor.

During the Silla Dynasty, Jinhae was an important port city and a major center for the production of salt and other goods. The city's strategic location allowed it to control the trade routes between the Silla Kingdom and other neighboring kingdoms.

In the 14th century, Jinhae was a major hub for the production of salt, which was a crucial commodity in ancient Korea. The city's salt production was so significant that it was known as the 'Salt Capital of Korea'.

During the Joseon Dynasty (1392 - 1910), Jinhae continued to be an important port city and a major center for trade and commerce. The city's harbor was expanded, and

## 데이터셋 로드 (GSM8K) Hugginface library

In [ ]:
from datasets import load_dataset

gsm8k = load_dataset("openai/gsm8k", "main")['test'] # For testing
gsm8k_train = load_dataset("openai/gsm8k", "main")['train'] # For examples

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [ ]:
print("Question:")
for l in gsm8k['question'][0].split("."):
    print(l)
print("="*100)
print("Answer:")
print(gsm8k['answer'][0])

Question:
Janet’s ducks lay 16 eggs per day
 She eats three for breakfast every morning and bakes muffins for her friends every day with four
 She sells the remainder at the farmers' market daily for $2 per fresh duck egg
 How much in dollars does she make every day at the farmers' market?
Answer:
Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18


# In-Context Learning (문맥 상 학습)

모델의 크기가 커질수록, 모델을 few-shot demonstration에서 배울 수 있는 능력이 생깁니다.

Zero shot < One shot < Few shot

### 문맥 상 학습으로 수학 문제 풀이

In [ ]:
### n-shot prompt를 만들어 보기!
### Train dataset에서 n 개의 예시를 랜덤으로 뽑아와서 shot으로 넣어보는 과정입니다.
import random

def construct_direct_prompt(num_exemplars: int): # num_exemplars: 사용할 shot의 개수
    sampled_indices = random.sample(range(len(gsm8k_train['question'])), num_exemplars) # "gsm8k_train"에서 random 모듈을 활용하여 랜덤하게 인덱스를 sampling하는 코드, 개수는 num_examplars 변수를 활용.

    instruction = "Instruction:\nSolve the following mathematical question and generate ONLY the answer after a tag, 'Answer:' without any rationale."


    # GSM8K에서 few-shot demonstration을 넣은 프롬프트를 작성하시면 됩니다.
    # 위 셀에 있는 GSM8K 데이터셋의 형태를 잘 보시면 됩니다.
    # Demonstration이 들어가는 형태는 원하는 형태로 바꿔서 사용하시면 됩니다.
    prompt = instruction
    for idx, i in enumerate(sampled_indices):
      cur_question = gsm8k_train['question'][i] # 데이터셋에서 예시로 넣을 질문 가져오기
      cur_answer = gsm8k_train['answer'][i].split("####")[-1].strip() # 데이터셋에서 위 질문의 정답 가져오기
      prompt += f"\n[Example {idx + 1}]\nQuestion:\n{cur_question}\nAnswer:{cur_answer}\n" # Few-shot 형태로 넣어주기

    prompt += f"\n[Example {num_exemplars+1}]\n"
    prompt += "Question:\n{question}\nAnswer:"

    return prompt

Example prompt

```
Instruction:
Solve the following mathematical question and generate ONLY the answer after a tag, 'Answer:' without any rationale.
[Example 1]
Question:
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
Answer:72

[Example 2]
Question:
Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?
Answer:10

[Example 3]
Question:
Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?
Answer:5

[Example 4]
Question: {question}
Answer:
```

In [ ]:
direct_prompt_1shot = construct_direct_prompt(1)
direct_prompt_3shot = construct_direct_prompt(3)

모델이 얼마나 잘 맞추는 지를 확인해 보겠습니다.

In [ ]:
prompt = direct_prompt_3shot # direct_prompt_1shot
print(prompt)

Instruction:
Solve the following mathematical question and generate ONLY the answer after a tag, 'Answer:' without any rationale.
[Example 1]
Question:
{question}
Answer:


In [ ]:
from tqdm import tqdm
import json
import re

results_collected = []
pass_collected = []
VERBOSE = False # 일부 결과만 미리 확인하고 싶을 때!
def parse_model_responses(text):
    if not text: return None

    # Enhancing regex to capture numbers possibly associated with units or other contexts
    regex = r"(?:Answer:|Model response:)\s*\$?([0-9,]+)\b|([0-9,]+)\s*(meters|cups|miles|minutes)"
    matches = re.finditer(regex, text, re.MULTILINE)
    results = [match.group(1) if match.group(1) else match.group(2).replace(",", "") for match in matches]

    # If no matches found in previous patterns, attempt to retrieve simpler numeric or monetary values
    if len(results) == 0:
        if "####" in text:  # Some example's finial answer format.
            val = text.split("####")[-1].strip()
            results.append(val.replace(",", ""))
        elif "Answer:" in text:
            val = text.split("Answer:")[-1].strip()
            results.append(val.replace(",", ""))
        else:
            return None

    if VERBOSE:
        print(f"Results found: {results}")
    return results[-1] if results else None

for i in tqdm(range(50)):
    cur_question = gsm8k['question'][i] # gsm8k에서 질문을 가져와야 합니다.
    cur_answer = gsm8k['answer'][i] # gsm8k에서 정답만 가져와야 합니다.
    # 문제을 프롬프트에 넣기
    cur_model_input = prompt.format(question=cur_question) # 최종 prompt 형태를 고려해서 prompt formatting하기!
    response = chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": "Follow the format and pattern of the provided Question-Answer examples."
            },
            {
                "role": "user",
                "content": cur_model_input,
            }
        ],
        model="llama-3.1-8b-instant",
        temperature=0
    )
    result = response.choices[0].message.content # 위에서 확인했던 chat_completion 모듈을 활용해서 모델의 응답을 얻어보겠습니다!

    if "Error:" in result:
        print(result)
        break

    cur_prediction = parse_model_responses(result)
    cur_answer = parse_model_responses(cur_answer)

    if VERBOSE:
        print("Raw response:", result)
        print("Predicted answer:",cur_prediction)
        print("Reference answer:",cur_answer)

    if cur_prediction is None or cur_answer is None:
        is_correct = False
    else:
        is_correct = (cur_prediction.strip().replace("$", "") == cur_answer.strip().replace("$", ""))

    pass_collected.append(is_correct)
    results_collected.append({"question": cur_question, "answer": cur_answer, "prediction": cur_prediction})
    print(f"Acc: {sum(pass_collected)/ len(pass_collected)}")

    if VERBOSE and i == 10:
        break

    ### 저장하고 싶은 형태의 이름으로 변경하여서 저장하여도 무방합니다!
    with open("result_3shot_direct.json", "w") as f:
        json.dump(results_collected, f, indent=4)

  4%|▍         | 2/50 [00:00<00:11,  4.21it/s]

Acc: 0.0
Acc: 0.0


  8%|▊         | 4/50 [00:00<00:09,  4.63it/s]

Acc: 0.0
Acc: 0.25


 10%|█         | 5/50 [00:01<00:09,  4.52it/s]

...

Acc: 0.4583333333333333


 98%|█████████▊| 49/50 [01:27<00:02,  2.30s/it]

Acc: 0.4489795918367347


100%|██████████| 50/50 [01:29<00:00,  1.80s/it]

Acc: 0.44


# Chain of Thoughts Prompting

## Few shot CoT

Generate intermediate reasoning steps by providing few-shot demonstrations

In [ ]:
def construct_cot_prompt(num_exemplars):
  sampled_indices = random.sample([i for i in range(len(gsm8k_train['question']))], num_exemplars)

  #CoT의 few-shot demonstration을 한번 만들어 보겠습니다!
  instruction = "Instruction:\nRefer to the following examples to understand the reasoning style and output format."
  prompt = instruction
  for idx, i in enumerate(sampled_indices):
      cur_question = gsm8k_train['question'][i] # 데이터셋에서 예시로 넣을 질문 가져오기
      cur_answer = gsm8k_train['answer'][i] # 데이터셋에서 위 질문의 정답 가져오기
      prompt += f"\n[Example {idx + 1}]\nQuestion:\n{cur_question}\nOutput:{cur_answer}\n"
      # Few-shot demonstration으로 추론 과정을 넣어주세요. 위의 코드 및 아래 예시를 참고하면서 진행하시면 됩니다.

  prompt += f"\n[Real Question] {num_exemplars+1}]\n"
  prompt += "Question:\n{question}\n"

  return prompt

Example prompt

```
Instruction:
Solve the following questions. Provide your final answer based on the rationale using an identifier '####'.

[Example 1]
Question:
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
Output:Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72

[Example 2]
Question:
Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?
Output:Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.
Working 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.
#### 10

[Example 3]
Question:
Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?
Output:In the beginning, Betty has only 100 / 2 = $<<100/2=50>>50.
Betty's grandparents gave her 15 * 2 = $<<15*2=30>>30.
This means, Betty needs 100 - 50 - 30 - 15 = $<<100-50-30-15=5>>5 more.
#### 5

[Example 4]
Question: {question}
```

In [ ]:
cot_prompt_1shot = construct_cot_prompt(1)
cot_prompt_3shot = construct_cot_prompt(3)

In [ ]:
prompt = cot_prompt_3shot # cot_prompt_1shot / cot_instruction_3shot
print(prompt)

Instruction:
Refer to the following examples to understand the reasoning style and output format.
[Example 1]
Question:
Toby is in a juggling contest with a friend. The winner is whoever gets the most objects rotated around in 4 minutes. Toby has 5 baseballs and each one makes 80 rotations. His friend has 4 apples and each one makes 101 rotations. How many total rotations of objects are made by the winner?
Output:Toby gets 400 full rotations because 5 x 80 = <<5*80=400>>400
His friend get 404 rotations because 4 x 101 = <<4*101=404>>404
The winner rotated 404 objects because 404 > 400
#### 404

[Example 2]
Question:
Dimitri eats 3 burgers per day. Each burger has a total of 20 calories. How many calories will he get after two days?
Output:The total number of calories he gets per day is 20 x 3 = <<20*3=60>>60.
Therefore the total number of calories he will get after 2 days is 60 x 2 = <<60*2=120>>120.
#### 120

[Example 3]
Question:
MIlle is making snack packs for her kindergarten class

Let's check the accuracy of the model!

In [ ]:
VERBOSE=False
results_collected = []
pass_collected = []

def parse_model_responses(text):
    if not text: return None

    # Enhancing regex to capture numbers possibly associated with units or other contexts
    regex = r"(?:Answer:|Model response:)\s*\$?([0-9,]+)\b|([0-9,]+)\s*(meters|cups|miles|minutes)"
    matches = re.finditer(regex, text, re.MULTILINE)
    results = [match.group(1) if match.group(1) else match.group(2).replace(",", "") for match in matches]

    # If no matches found in previous patterns, attempt to retrieve simpler numeric or monetary values
    if len(results) == 0:
        if "####" in text:  # Some example's finial answer format.
            val = text.split("####")[-1].strip()
            results.append(val.replace(",", ""))
        elif "Answer:" in text:
            val = text.split("Answer:")[-1].strip()
            results.append(val.replace(",", ""))
        else:
            return None

    if VERBOSE:
        print(f"Results found: {results}")
    return results[-1] if results else None

for i in tqdm(range(50)):
    cur_question = gsm8k['question'][i]
    cur_answer = gsm8k['answer'][i]
    cur_model_input = prompt.format(question=cur_question)
    result = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a math solver. Solve the problem step-by-step. "
                    "Make sure to strictly follow the format. "
                    "End your answer with '#### [Answer]'."
                )
            },
            {
                "role": "user",
                "content": cur_model_input
            }
        ],
        model="llama-3.1-8b-instant",
        temperature=0
    ).choices[0].message.content

    if "Error:" in result:
        print(result)
        break

    cur_prediction = parse_model_responses(result)
    cur_answer = parse_model_responses(cur_answer)

    if VERBOSE:
        print("Raw response:", result)
        print("Predicted answer:",cur_prediction)
        print("Reference answer:",cur_answer)


    if cur_prediction is None or cur_answer is None:
        is_correct = False
    else:
        is_correct = (cur_prediction.strip().replace("$", "") == cur_answer.strip().replace("$", ""))

    pass_collected.append(is_correct)
    results_collected.append({"question": cur_question, "answer": cur_answer, "prediction": cur_prediction})
    print(f"Acc: {sum(pass_collected)/ len(pass_collected)}")

    if VERBOSE and i == 10:
        break


    with open("result_3shot_cot.json", "w") as f:
        json.dump(results_collected, f, indent=4)

  2%|▏         | 1/50 [00:04<03:44,  4.59s/it]

Acc: 1.0


  4%|▍         | 2/50 [00:12<05:03,  6.32s/it]

Acc: 1.0


  6%|▌         | 3/50 [00:19<05:24,  6.91s/it]

...

Acc: 0.8275862068965517


 60%|██████    | 30/50 [03:57<01:50,  5.54s/it]

Acc: 0.8333333333333334


 60%|██████    | 30/50 [04:19<02:52,  8.64s/it]


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kaw8r5pyfypadp77b025h7t2` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 5618, Requested 843. Please try again in 4.61s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

## Zero shot CoT



In [ ]:
def construct_0shot_cot():
  # instruction은 Few-shot CoT에서 작성하셨던 prompt를 생각하면서 작성해봅시다.
  instruction = "Let's think step by step"
  # "Let's think step by step through simple pseudo code."
  prompt = instruction

  prompt += "Question:\n{question}\n"

  return prompt

어떻게 추론 과정과 정답을 추출해 낼 수 있을까요?

In [ ]:
prompt = construct_0shot_cot()
print(prompt)

Let's think step by step through simple pseudo code.Question:
{question}



In [ ]:
import time

VERBOSE=False # If you want to check some of the results first
results_collected = []
pass_collected = []

for i in tqdm(range(50)):
    cur_question = gsm8k['question'][i]
    cur_answer = gsm8k['answer'][i]
    cur_model_input = prompt.format(question=cur_question)
    ### Zero-shot CoT를 위한 응답 생성 chat completion을 작성해 보겠습니다! 어떻게 completion 작성해야 할 지 고민해 보시면서 작성해 주세요.
    ### LLM이 답변을 내놓을 때에는 "role": "assistant"를 활용하여 작성합니다.
    messages=[
        {
            "role": "system",
            "content": "You are a math solver."
        },
        {
            "role": "user",
            "content": cur_model_input
        }
    ]

    reason = client.chat.completions.create(
        messages=messages,
        model="llama-3.1-8b-instant",
        temperature=0,
        max_tokens=1024
    ).choices[0].message.content

    messages.append({
        "role": "assistant",
        "content": reason
    })

    messages.append({
        "role": "user",
        "content": ("Based on the reasoning above, provide the final answer. "
                    "Strictly follow this format: '#### [Answer]'. "
                    "For example, if the answer is 12, output: '#### 12'.")
    })

    result = client.chat.completions.create(
        messages=messages,
        model="llama-3.1-8b-instant",
        temperature=0,
        max_tokens=1024
    ).choices[0].message.content

    if "Error:" in result:
        print(result)
        break

    cur_prediction = parse_model_responses(result)
    cur_answer = parse_model_responses(cur_answer)

    cur_prediction = parse_model_responses(result)
    if VERBOSE:
        print("Raw response:", result)
        print("Predicted answer:",cur_prediction)
        print("Reference answer:",cur_answer)


    if cur_prediction is None or cur_answer is None:
        is_correct = False
    else:
        is_correct = (cur_prediction.strip().replace("$", "") == cur_answer.strip().replace("$", ""))

    pass_collected.append(is_correct)
    results_collected.append({"question": cur_question, "answer": cur_answer, "reason": reason, "prediction": cur_prediction})
    print(f"Acc: {sum(pass_collected)/ len(pass_collected)}")

    if VERBOSE and i == 10:
        break

    with open("result_0shot_cot.json", "w") as f:
        json.dump(results_collected, f, indent=4)

    time.sleep(10)

  0%|          | 0/50 [00:00<?, ?it/s]

Acc: 1.0


  2%|▏         | 1/50 [00:10<08:56, 10.95s/it]

Acc: 1.0


  4%|▍         | 2/50 [00:21<08:35, 10.75s/it]

...

Acc: 0.9473684210526315


 38%|███▊      | 19/50 [03:26<05:35, 10.84s/it]

Acc: 0.95


 40%|████      | 20/50 [03:52<05:49, 11.64s/it]


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kaw8r5pyfypadp77b025h7t2` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 4364, Requested 1650. Please try again in 140ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}